In [1]:
import pyreth

# Create PyReth instance (not named 'reth' as per your request)
pyreth_client = pyreth.PyReth()
processor = pyreth_client.tx_processor()
simulator = pyreth_client.simulator()
chain_query = pyreth_client.chain_query()


In [ ]:
# Transaction 1: The one with LackOfFundForMaxFee error
tx1_hash = "0x6e115e08f7832ba8291e507c3b44b529163ef1302cba8aa39a433ba700e2b214"
tx1_account = "0xcC228F9F1428314acc460e7Da110C64415A2F05a"

print("\n1. INVESTIGATING TX WITH LACKOFFUNDFEE ERROR")
print("-" * 40)
print(f"TX Hash: {tx1_hash}")
print(f"Account: {tx1_account}")

# Process the actual transaction
try:
    print("\nProcessing actual transaction...")
    tx1_processed = processor.process_transaction(tx1_hash)
    print(f"  Block: {tx1_processed.block_number}")
    print(f"  From: {tx1_processed.from_address}")
    print(f"  To: {tx1_processed.to_address}")
    print(f"  Value: {tx1_processed.value}")
    
    # Get fees information
    fees = tx1_processed.fees  # It's a property, not a method
    print(f"  Gas Used: {fees['gas_used']}")
    print(f"  Gas Price: {fees['gas_price']}")
    print(f"  Status: {tx1_processed.status}")
    
    # Check balance at transaction block
    print(f"\nChecking account balance at block {tx1_processed.block_number}...")
    balance_at_tx = chain_query.get_balance(tx1_account, tx1_processed.block_number)
    print(f"  Balance at TX block: {balance_at_tx} wei")
    
    # Check balance at current state
    print("\nChecking account balance at current state...")
    current_balance = chain_query.get_balance(tx1_account)
    print(f"  Current balance: {current_balance} wei")
    
    # Try to simulate at the transaction's block
    print(f"\nSimulating transaction at its original block {tx1_processed.block_number}...")
    try:
        # Build transaction for simulation
        sim_tx = {
            "from": tx1_processed.from_address,
            "to": tx1_processed.to_address,
            "value": str(tx1_processed.value),
            "data": tx1_processed.input,
            "gas": fees['gas_used'] * 2,  # Use a reasonable gas limit
        }
        
        sim_result = simulator.simulate_transaction(sim_tx, tx1_processed.block_number)
        print(f"  Simulation successful!")
        sim_fees = sim_result.fees
        print(f"  Gas used: {sim_fees['gas_used']}")
        print(f"  Status: {sim_result.status}")
    except Exception as e:
        print(f"  Simulation failed: {e}")
        
except Exception as e:
    print(f"  Error processing transaction: {e}")

# Transaction 2: The undetected scam
tx2_hash = "0xb20e91c60b35647725b1878b60e2ccf6543fc17983983227656cf98bebb22966"
tx2_pool = "0xd5113065d0dA0CD94F8c0Ba7B2Fa61d8A48AE404"
tx2_creator = "0x7D804D810147e0297b97f11b6f017147DFaE2fc2"

print("\n\n2. INVESTIGATING UNDETECTED SCAM TRANSACTION")
print("-" * 40)
print(f"TX Hash: {tx2_hash}")
print(f"Pool: {tx2_pool}")
print(f"Creator: {tx2_creator}")

try:
    print("\nProcessing actual transaction...")
    tx2_processed = processor.process_transaction(tx2_hash)
    print(f"  Block: {tx2_processed.block_number}")
    print(f"  From: {tx2_processed.from_address}")
    print(f"  To: {tx2_processed.to_address}")
    fees2 = tx2_processed.fees
    print(f"  Gas Used: {fees2['gas_used']}")
    print(f"  Status: {tx2_processed.status}")
    
    # Analyze events
    print(f"\nTransaction Events:")
    print(f"  ERC20 Transfers: {len(tx2_processed.erc20_transfers)}")
    print(f"  Uniswap V2 Events: {len(tx2_processed.uniswap_v2_swaps)}")
    print(f"  Internal Transactions: {len(tx2_processed.internal_transactions)}")
    
    # Look for liquidity removal
    for event in tx2_processed.uniswap_v2_swaps:
        if event.get('type') == 'burn':
            print(f"\n  LIQUIDITY REMOVAL DETECTED:")
            print(f"    Pool: {event.get('pair_address')}")
            print(f"    Amount0: {event.get('amount0')}")
            print(f"    Amount1: {event.get('amount1')}")
    
    # Check pool state before and after
    print(f"\nChecking pool ETH balance...")
    print(f"  At TX block {tx2_processed.block_number}:")
    pool_balance_at_tx = chain_query.get_balance(tx2_pool, tx2_processed.block_number)
    print(f"    Pool balance: {int(pool_balance_at_tx) / 10**18:.6f} ETH")
    
    # Check one block after
    pool_balance_after = chain_query.get_balance(tx2_pool, tx2_processed.block_number + 1)
    print(f"  After TX (block {tx2_processed.block_number + 1}):")
    print(f"    Pool balance: {int(pool_balance_after) / 10**18:.6f} ETH")
    
    balance_change = (int(pool_balance_at_tx) - int(pool_balance_after)) / 10**18
    print(f"  Balance removed: {balance_change:.6f} ETH")
    
except Exception as e:
    print(f"  Error processing transaction: {e}")

# Get latest block for reference
try:
    latest_block = simulator.get_latest_block()
    print(f"\n\nLatest block in database: {latest_block}")
except Exception as e:
    print(f"\n\nCould not get latest block: {e}")

print("\n" + "=" * 80)
print("INVESTIGATION COMPLETE")
print("=" * 80)



1. INVESTIGATING TX WITH LACKOFFUNDFEE ERROR
----------------------------------------
TX Hash: 0x6e115e08f7832ba8291e507c3b44b529163ef1302cba8aa39a433ba700e2b214
Account: 0xcC228F9F1428314acc460e7Da110C64415A2F05a

Processing actual transaction...
  Block: 23173855
  From: 0xcC228F9F1428314acc460e7Da110C64415A2F05a
  To: 0x7a250d5630B4cF539739dF2C5dAcb4c659F2488D
  Value: 0
  Gas Used: 180384
  Gas Price: 397254292
  Status: 1

Checking account balance at block 23173855...
  Balance at TX block: 1759319667191209581 wei

Checking account balance at current state...
  Current balance: 0 wei

Simulating transaction at its original block 23173855...
  Simulation successful!
  Gas used: 29437
  Status: failed


2. INVESTIGATING UNDETECTED SCAM TRANSACTION
----------------------------------------
TX Hash: 0xb20e91c60b35647725b1878b60e2ccf6543fc17983983227656cf98bebb22966
Pool: 0xd5113065d0dA0CD94F8c0Ba7B2Fa61d8A48AE404
Creator: 0x7D804D810147e0297b97f11b6f017147DFaE2fc2

Processing actual t

In [ ]:
# Get trading simulator
trading_sim = pyreth_client.trading_simulator()

# Process transaction to get ProcessedTransaction
tx_processed = processor.process_transaction("0x57acab49778bb8c38efb5261e2b3473feca6c018856f0557f9a3f4d976355745")

# Simulate trading sequence
result = trading_sim.simulate_tx_with_buy_sell_seq(
    prior_tx=tx_processed,  # Optional ProcessedTransaction that affects trading
    token_address="0x6982508145454Ce325dDbE47a25d4ec3d2311933",  # PEPE token
    pool_address="0xa43fe16908251ee70ef74718545e4fe6c5ccec9f",   # PEPE/WETH pool
    block_number=None  # Use latest block (default)
)

# Access results - full ProcessedTransaction objects
print(f"Trading enabled: {result.trading_enabled}")
print(f"Buy tax: {result.buy_tax}%")
print(f"Sell tax: {result.sell_tax}%")
print(f"Buy transaction: {result.buy_tx}")  # Full ProcessedTransaction
print(f"Sell transaction: {result.sell_tx}")  # Full ProcessedTransaction


RuntimeError: Simulation failed: EVM error: Transaction(LackOfFundForMaxFee { fee: 88581432793248, balance: 0 })

In [7]:
tx_processed

ProcessedTransaction(hash='0x57acab49778bb8c38efb5261e2b3473feca6c018856f0557f9a3f4d976355745', block_number=23190296, block_timestamp=1755789407, txn_index=338, from_address='0x5DE2C12FCC0D2084DDF46E062CA6808CC997B8EC', to_address='0x0XF068C6D0390797E622B668531789ADB31FF2A3C0', contract_address=None, value=0.0, status=True, nonce=3, txn_type='approval', actions=["token_approval"], fees=TransactionFees(gas_price=1920589584, gas_used=46122, txn_fee=0.000088581432793248, protocol_type='unknown', max_fee_per_gas=None, max_priority_fee=None), bribe_amount=0, unique_addresses={'0x5DE2C12FCC0D2084DDF46E062CA6808CC997B8EC', '0xF068C6D0390797E622B668531789ADB31FF2A3C0', '0x7A250D5630B4CF539739DF2C5DACB4C659F2488D'}, erc20_contracts={'0xF068C6D0390797E622B668531789ADB31FF2A3C0'}, eth_transfers=[], erc20_transfers=[], erc721_transfers=[], erc1155_transfers=[], internal_transactions=[InternalTransaction(from_address='0x5DE2C12FCC0D2084DDF46E062CA6808CC997B8EC', to_address='0xF068C6D0390797E622B66

In [5]:
result.prior_tx